In [2]:
import pandas as pd

In [3]:
import re
import json

def fix_entities_format(text):
    if pd.isna(text):
        return []

    # Add missing commas between dicts
    text = re.sub(r"}\s*{", "}, {", text)

    # Replace single quotes with double quotes (for JSON)
    text = text.replace("'", '"')

    # Wrap properly if needed
    if not text.startswith("["):
        text = "[" + text
    if not text.endswith("]"):
        text = text + "]"

    try:
        return json.loads(text)
    except:
        return []

In [4]:
import pandas as pd

df = pd.read_csv("ner_dataset.csv")

df["entities"] = df["entities"].apply(fix_entities_format)

In [5]:
print(df["entities"].iloc[0])

[{'class': 'ORGANISM', 'end': 4, 'start': 0}, {'class': 'ORGANISM', 'end': 9, 'start': 5}, {'class': 'CHEMICALS', 'end': 30, 'start': 26}, {'class': 'LOCATION', 'end': 40, 'start': 31}, {'class': 'ACTIVITY', 'end': 60, 'start': 45}, {'class': 'CHEMICALS', 'end': 80, 'start': 66}, {'class': 'ORGANISM', 'end': 91, 'start': 85}, {'class': 'CHEMICALS', 'end': 119, 'start': 103}, {'class': 'ACTIVITY', 'end': 135, 'start': 120}, {'class': 'ACTIVITY', 'end': 151, 'start': 141}, {'class': 'CHEMICALS', 'end': 226, 'start': 222}, {'class': 'FUNCTION', 'end': 245, 'start': 227}, {'class': 'CHEMICALS', 'end': 285, 'start': 281}, {'class': 'ORGANISM', 'end': 364, 'start': 360}, {'class': 'ORGANISM', 'end': 370, 'start': 365}, {'class': 'LOCATION', 'end': 441, 'start': 433}, {'class': 'LOCATION', 'end': 454, 'start': 449}, {'class': 'CHEMICALS', 'end': 475, 'start': 471}, {'class': 'CHEMICALS', 'end': 488, 'start': 487}, {'class': 'CHEMICALS', 'end': 508, 'start': 489}, {'class': 'ORGANISM', 'end': 

# 2. Load + Fix Dataset

In [6]:
import pandas as pd
import ast

df = pd.read_csv("ner_dataset.csv")

# Safe parser (handles both string + list)
def safe_parse(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return x

df["entities"] = df["entities"].apply(safe_parse)

print(type(df["entities"].iloc[0]))  # should be list

<class 'list'>


# 3. Tokenization + BIO Tagging

In [7]:
import re

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text)

def create_bio_tags(text, entities):
    tokens = tokenize(text)
    tags = ["O"] * len(tokens)

    spans = []
    idx = 0

    for token in tokens:
        start = text.find(token, idx)
        end = start + len(token)
        spans.append((start, end))
        idx = end

    for ent in entities:
        ent_start = ent["start"]
        ent_end = ent["end"]
        label = ent["class"]

        for i, (start, end) in enumerate(spans):
            if start >= ent_start and end <= ent_end:
                if start == ent_start:
                    tags[i] = f"B-{label}"
                else:
                    tags[i] = f"I-{label}"

    return tokens, tags

# 4. Prepare Dataset

In [8]:
sentences = []
tags_list = []

for _, row in df.iterrows():
    tokens, tags = create_bio_tags(row["text"], row["entities"])
    sentences.append(tokens)
    tags_list.append(tags)

print(sentences[0])
print(tags_list[0])

['Weed', 'seed', 'inactivation', 'in', 'soil', 'mesocosms', 'via', 'biosolarization', 'with', 'mature', 'compost', 'and', 'tomato', 'processing', 'waste', 'amendments', 'Biosolarization', 'is', 'a', 'fumigation', 'alternative', 'that', 'combines', 'passive', 'solar', 'heating', 'with', 'amendment', '-', 'driven', 'soil', 'microbial', 'activity', 'to', 'temporarily', 'create', 'antagonistic', 'soil', 'conditions', ',', 'such', 'as', 'elevated', 'temperature', 'and', 'acidity', ',', 'that', 'can', 'inactivate', 'weed', 'seeds', 'and', 'other', 'pest', 'propagules', '.', 'The', 'aim', 'of', 'this', 'study', 'was', 'to', 'use', 'a', 'mesocosm', '-', 'based', 'field', 'trial', 'to', 'assess', 'soil', 'heating', ',', 'pH', ',', 'volatile', 'fatty', 'acid', 'accumulation', 'and', 'weed', 'seed', 'inactivation', 'during', 'biosolarization', '.', 'Biosolarization', 'for', '8', 'days', 'using', '2', '%', 'mature', 'green', 'waste', 'compost', 'and', '2', 'or', '5', '%', 'tomato', 'processing', '

In [9]:
from collections import defaultdict

word2idx = defaultdict(lambda: len(word2idx))
char2idx = defaultdict(lambda: len(char2idx))

label_set = set(tag for tags in tags_list for tag in tags)

tag2idx = {tag: i for i, tag in enumerate(label_set)}
idx2tag = {i: t for t, i in tag2idx.items()}

for sent in sentences:
    for word in sent:
        word2idx[word]
        for ch in word:
            char2idx[ch]

In [10]:
import torch

def encode(sent, tags):
    word_ids = [word2idx[w] for w in sent]
    char_ids = [[char2idx[c] for c in w] for w in sent]
    tag_ids = [tag2idx[t] for t in tags]

    return (
        torch.tensor(word_ids).unsqueeze(0),
        char_ids,
        torch.tensor(tag_ids).unsqueeze(0)
    )

In [11]:
import torch.nn as nn
from TorchCRF import CRF

class NERModel(nn.Module):
    def __init__(self, vocab_size, char_size, tagset_size,
                 word_emb_dim=100, char_emb_dim=30, hidden_dim=128):
        super().__init__()

        self.word_emb = nn.Embedding(vocab_size, word_emb_dim)

        self.char_emb = nn.Embedding(char_size, char_emb_dim)
        self.char_lstm = nn.LSTM(char_emb_dim, char_emb_dim // 2,
                                 bidirectional=True, batch_first=True)

        self.lstm = nn.LSTM(word_emb_dim + char_emb_dim,
                            hidden_dim // 2,
                            bidirectional=True,
                            batch_first=True)

        self.fc = nn.Linear(hidden_dim, tagset_size)
        self.crf = CRF(tagset_size)

    def get_char_rep(self, char_ids):
        reps = []
        for word_chars in char_ids:
            chars = torch.tensor(word_chars).unsqueeze(0)
            emb = self.char_emb(chars)
            _, (hidden, _) = self.char_lstm(emb)
            rep = torch.cat((hidden[0], hidden[1]), dim=1)
            reps.append(rep.squeeze(0))
        return torch.stack(reps)

    def forward(self, word_ids, char_ids, tags=None):
        word_emb = self.word_emb(word_ids)

        char_rep = self.get_char_rep(char_ids).unsqueeze(0)
        combined = torch.cat((word_emb, char_rep), dim=2)

        lstm_out, _ = self.lstm(combined)
        emissions = self.fc(lstm_out)

        mask = torch.ones(emissions.size(0), emissions.size(1), dtype=torch.uint8)

        if tags is not None:
            return self.crf.forward(emissions, tags, mask)
        else:
            return self.crf.viterbi_decode(emissions, mask)

In [12]:
model = NERModel(len(word2idx), len(char2idx), len(tag2idx))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    total_loss = 0

    for sent, tags in zip(sentences[:300], tags_list[:300]):  # limit for speed
        word_ids, char_ids, tag_ids = encode(sent, tags)

        optimizer.zero_grad()
        loss = model(word_ids, char_ids, tag_ids)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss}")

C:\Users\Ayush Sah\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\TorchCRF\__init__.py:173: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorCompare.cpp:615.)
  score = torch.where(mask_t, score_t, score)


Epoch 0, Loss: -4621547.35647583
Epoch 1, Loss: -12315699.233398438
Epoch 2, Loss: -19058656.350585938
Epoch 3, Loss: -25618648.52734375
Epoch 4, Loss: -32102411.98828125


In [13]:
from seqeval.metrics import classification_report, f1_score

true_labels = []
pred_labels = []

for sent, tags in zip(sentences[:100], tags_list[:100]):
    word_ids, char_ids, _ = encode(sent, tags)
    preds = model(word_ids, char_ids)

    pred_tags = [idx2tag[p] for p in preds[0]]

    true_labels.append(tags)
    pred_labels.append(pred_tags)

print("F1 Score:", f1_score(true_labels, pred_labels))
print(classification_report(true_labels, pred_labels))

F1 Score: 0.0


C:\Users\Ayush Sah\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

     PRODUCT       0.00      0.00      0.00         0

   micro avg       0.00      0.00      0.00         0
   macro avg       0.00      0.00      0.00         0
weighted avg       0.00      0.00      0.00         0



In [14]:
text = df["text"].iloc[0]

tokens, _ = create_bio_tags(text, [])
word_ids, char_ids, _ = encode(tokens, ["O"] * len(tokens))

preds = model(word_ids, char_ids)

for token, pred in zip(tokens, preds[0]):
    print(token, idx2tag[pred])

Weed B-PRODUCT
seed B-PRODUCT
inactivation B-PRODUCT
in B-PRODUCT
soil B-PRODUCT
mesocosms B-PRODUCT
via B-PRODUCT
biosolarization B-PRODUCT
with B-PRODUCT
mature B-PRODUCT
compost B-PRODUCT
and B-PRODUCT
tomato B-PRODUCT
processing B-PRODUCT
waste B-PRODUCT
amendments B-PRODUCT
Biosolarization B-PRODUCT
is B-PRODUCT
a B-PRODUCT
fumigation B-PRODUCT
alternative B-PRODUCT
that B-PRODUCT
combines B-PRODUCT
passive B-PRODUCT
solar B-PRODUCT
heating B-PRODUCT
with B-PRODUCT
amendment B-PRODUCT
- B-PRODUCT
driven B-PRODUCT
soil B-PRODUCT
microbial B-PRODUCT
activity B-PRODUCT
to B-PRODUCT
temporarily B-PRODUCT
create B-PRODUCT
antagonistic B-PRODUCT
soil B-PRODUCT
conditions B-PRODUCT
, B-PRODUCT
such B-PRODUCT
as B-PRODUCT
elevated B-PRODUCT
temperature B-PRODUCT
and B-PRODUCT
acidity B-PRODUCT
, B-PRODUCT
that B-PRODUCT
can B-PRODUCT
inactivate B-PRODUCT
weed B-PRODUCT
seeds B-PRODUCT
and B-PRODUCT
other B-PRODUCT
pest B-PRODUCT
propagules B-PRODUCT
. B-PRODUCT
The B-PRODUCT
aim B-PRODUCT